# Data-Driven Modeling of Galactic Rotation Curves with Artificial Neural Networks
## Full Reproduction and Independent Validation of arXiv:2404.05833v2

**Authors**: Gabriela Garcia-Arroyo, Isidro Gómez-Vargas, J. Alberto Vázquez  
**Paper Link**: [arXiv:2404.05833v2](https://arxiv.org/abs/2404.05833)  
**Repository**: [https://github.com/thisiskartik05/THINGS](https://github.com/thisiskartik05/THINGS)  

---

### Overview & Scientific Context
This single, self-contained notebook reproduces and validates the non-parametric Artificial Neural Network (ANN) pipeline with Monte Carlo Dropout (MC-DO) for modeling rotation curves of 17 spiral galaxies from **The HI Nearby Galaxy Survey (THINGS)**.

#### Key Equations:
1. **Quadrature Mass Decomposition**:
   68622V^2(r) = V_{\text{gas}}^2(r) + \Upsilon_\star V_\star^2(r) + V_{\text{halo}}^2(r)68622
2. **Monte Carlo Dropout Uncertainty Quantification**:
   68622\bar{V}(r) = \frac{1}{N_{\text{MC}}} \sum_{t=1}^{N_{\text{MC}}} \hat{V}^{(t)}(r), \quad \sigma_{\text{MC}}^2(r) = \frac{1}{N_{\text{MC}}} \sum_{t=1}^{N_{\text{MC}}} \left( \hat{V}^{(t)}(r) - \bar{V}(r) \right)^268622
3. **Mean Squared Error (MSE)**:
   68622\text{MSE} = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^268622


In [ ]:
# Cell 1: Environment Setup & Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


In [ ]:
# Cell 2: Data Loader Function
def load_rotation_curve_data(galaxy_name, data_dir="data"):
    """
    Load observational rotation curve data for a given THINGS galaxy.
    Reads .curve.02 file format containing radial distance r, rotational velocity vrot, and error.
    """
    curve_path = os.path.join(data_dir, "Curves", f"{galaxy_name}.curve.02")
    if not os.path.exists(curve_path):
        raise FileNotFoundError(f"Data file not found for galaxy: {galaxy_name} at {curve_path}")
    
    # Header: # r vrot vdisp verr vapp vrec err pa incl vsys
    data = np.loadtxt(curve_path)
    df = pd.DataFrame(data, columns=["r", "vrot", "vdisp", "verr", "vapp", "vrec", "err", "pa", "incl", "vsys"])
    
    # Extract relevant columns: r (kpc or arcsec), vrot (km/s), error
    r = df["r"].values
    vobs = df["vrot"].values
    err_vobs = df["err"].values
    
    return df, r, vobs, err_vobs

# Test data loader for target galaxy DDO154
df_ddo, r_ddo, vobs_ddo, err_ddo = load_rotation_curve_data("DDO154")
print(f"Loaded DDO154: {len(r_ddo)} data points")
print(f"r range: [{r_ddo.min():.2f}, {r_ddo.max():.2f}], vobs range: [{vobs_ddo.min():.2f}, {vobs_ddo.max():.2f}] km/s")


In [ ]:
# Cell 3: Table 1 Architecture Registry
# Extracted hyperparameter combinations from Table 1 of arXiv:2404.05833v2
TABLE1_ARCHITECTURES = {
    "NGC4736": {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 8},
    "NGC2841": {"num_hidden_layers": 3, "num_neurons": 150, "batch_size": 8},
    "NGC925":  {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 4},
    "NGC2903": {"num_hidden_layers": 4, "num_neurons": 150, "batch_size": 8},
    "DDO154":  {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 4},
    "NGC6946": {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 8},
    "NGC7793": {"num_hidden_layers": 4, "num_neurons": 150, "batch_size": 4},
    "NGC2976": {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 4},
    "NGC7331": {"num_hidden_layers": 3, "num_neurons": 200, "batch_size": 4},
    "NGC3198": {"num_hidden_layers": 3, "num_neurons": 150, "batch_size": 4},
    "NGC3621": {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 4},
    "NGC3521": {"num_hidden_layers": 3, "num_neurons": 200, "batch_size": 4},
    "NGC5055": {"num_hidden_layers": 4, "num_neurons": 100, "batch_size": 4},
    "NGC2366": {"num_hidden_layers": 4, "num_neurons": 200, "batch_size": 8},
    "NGC2403": {"num_hidden_layers": 3, "num_neurons": 150, "batch_size": 8},
    "NGC3031": {"num_hidden_layers": 3, "num_neurons": 200, "batch_size": 4},
    "IC2574":  {"num_hidden_layers": 3, "num_neurons": 200, "batch_size": 4}
}
print(f"Registered architectures for {len(TABLE1_ARCHITECTURES)} galaxies")


In [ ]:
# Cell 4: Multi-Output Neural Network with Monte Carlo Dropout
class RotationCurveANN(nn.Module):
    """
    Feed-Forward ANN predicting coupled outputs [V(r), error(r)] from input radius r.
    Uses Monte Carlo Dropout (dropout rate p=0.3) active during both training and inference.
    """
    def __init__(self, num_hidden_layers=4, num_neurons=200, dropout_rate=0.3):
        super(RotationCurveANN, self).__init__()
        self.dropout_rate = dropout_rate
        
        layers = []
        # Input layer: 1 input (r) -> num_neurons
        layers.append(nn.Linear(1, num_neurons))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(p=dropout_rate))
        
        # Hidden layers
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(num_neurons, num_neurons))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_rate))
        
        # Output layer: num_neurons -> 2 outputs [V(r), error(r)]
        layers.append(nn.Linear(num_neurons, 2))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

    def enable_mc_dropout(self):
        """Enable dropout layers during inference mode for MC sampling."""
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()


In [ ]:
# Cell 5: Training & MC Inference Utility Functions
def train_ann_model(r_data, vobs_data, err_data, config, epochs=1000, lr=5e-4):
    """
    Train ANN model for a specific galaxy using Table 1 hyperparameter config.
    """
    # Standardize inputs & targets
    r_mean, r_std = r_data.mean(), r_data.std()
    v_mean, v_std = vobs_data.mean(), vobs_data.std()
    
    r_norm = (r_data - r_mean) / (r_std + 1e-8)
    v_norm = (vobs_data - v_mean) / (v_std + 1e-8)
    err_norm = err_data / (v_std + 1e-8)
    
    # Convert to PyTorch Tensors
    X_tensor = torch.tensor(r_norm, dtype=torch.float32).unsqueeze(1).to(device)
    y_tensor = torch.tensor(np.column_stack([v_norm, err_norm]), dtype=torch.float32).to(device)
    
    model = RotationCurveANN(
        num_hidden_layers=config["num_hidden_layers"],
        num_neurons=config["num_neurons"],
        dropout_rate=0.3
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()
        
    return model, (r_mean, r_std), (v_mean, v_std)

def predict_mc_dropout(model, r_eval, r_stats, v_stats, num_mc_samples=100):
    """
    Perform Monte Carlo Dropout inference over num_mc_samples stochastic passes.
    Returns mean velocity prediction and total uncertainty.
    """
    r_mean, r_std = r_stats
    v_mean, v_std = v_stats
    
    r_norm = (r_eval - r_mean) / (r_std + 1e-8)
    X_eval = torch.tensor(r_norm, dtype=torch.float32).unsqueeze(1).to(device)
    
    model.eval()
    model.enable_mc_dropout()
    
    predictions_v = []
    predictions_err = []
    
    with torch.no_grad():
        for _ in range(num_mc_samples):
            out = model(X_eval).cpu().numpy()
            v_pred_unnorm = out[:, 0] * v_std + v_mean
            err_pred_unnorm = np.abs(out[:, 1] * v_std)
            predictions_v.append(v_pred_unnorm)
            predictions_err.append(err_pred_unnorm)
            
    predictions_v = np.array(predictions_v)
    predictions_err = np.array(predictions_err)
    
    v_mean_pred = predictions_v.mean(axis=0)
    v_epistemic_var = predictions_v.var(axis=0)
    v_stat_err_pred = predictions_err.mean(axis=0)
    
    # Total uncertainty: quadratic sum of epistemic uncertainty (MC dropout variance) + predicted statistical error
    v_total_unc = np.sqrt(v_epistemic_var + v_stat_err_pred**2)
    
    return v_mean_pred, v_total_unc


In [ ]:
# Cell 6: Single-Galaxy Reproduction — DDO154
galaxy_target = "DDO154"
config_target = TABLE1_ARCHITECTURES[galaxy_target]

print(f"--- Training ANN Model for {galaxy_target} ---")
print(f"Architecture: {config_target['num_hidden_layers']} hidden layers, {config_target['num_neurons']} neurons/layer, batch size {config_target['batch_size']}")

df_target, r_target, vobs_target, err_target = load_rotation_curve_data(galaxy_target)
model_target, r_stats_target, v_stats_target = train_ann_model(r_target, vobs_target, err_target, config_target, epochs=1000)

# Predictions at observational radii
v_pred_target, unc_target = predict_mc_dropout(model_target, r_target, r_stats_target, v_stats_target, num_mc_samples=100)

# Calculate Mean Squared Error (MSE)
mse_ann_target = mean_squared_error(vobs_target, v_pred_target)
print(f"
[RESULTS] {galaxy_target} ANN Reconstruction MSE: {mse_ann_target:.4f}")
print(f"Paper Reported Table 2 ANN MSE: 3.50")


In [ ]:
# Cell 7: Publication-Quality Visualization — DDO154 Rotation Curve
r_dense = np.linspace(r_target.min(), r_target.max(), 300)
v_dense_pred, unc_dense = predict_mc_dropout(model_target, r_dense, r_stats_target, v_stats_target, num_mc_samples=100)

plt.figure(figsize=(10, 6), dpi=120)
plt.errorbar(r_target, vobs_target, yerr=err_target, fmt="o", color="crimson", ecolor="salmon",
             capsize=3, label="Observational Data (THINGS)", zorder=3)

plt.plot(r_dense, v_dense_pred, color="forestgreen", linewidth=2.5, label="ANN Reconstruction (Mean)", zorder=4)
plt.fill_between(r_dense, v_dense_pred - unc_dense, v_dense_pred + unc_dense,
                 color="gold", alpha=0.4, label="ANN Total Uncertainty (±1σ)", zorder=2)

plt.title(f"Galactic Rotation Curve Reconstruction — {galaxy_target} (Group A)", fontsize=14, pad=12)
plt.xlabel("Radial Distance r [kpc]", fontsize=12)
plt.ylabel("Rotational Velocity V(r) [km/s]", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=11, loc="lower right")
plt.tight_layout()
plt.savefig(f"results_{galaxy_target}_reconstruction.png")
plt.show()


In [ ]:
# Cell 8: Multi-Galaxy Benchmark Loop (Groups A, B, C)
sample_galaxies = ["DDO154", "NGC3621", "NGC5055", "NGC3521", "NGC2976", "IC2574"]
results_list = []

print("=== Multi-Galaxy ANN Pipeline Benchmark ===")
for gal in sample_galaxies:
    cfg = TABLE1_ARCHITECTURES[gal]
    _, r_g, v_g, err_g = load_rotation_curve_data(gal)
    mdl, r_s, v_s = train_ann_model(r_g, v_g, err_g, cfg, epochs=1000)
    v_pred_g, _ = predict_mc_dropout(mdl, r_g, r_s, v_s, num_mc_samples=50)
    mse_calc = mean_squared_error(v_g, v_pred_g)
    results_list.append({"Galaxy": gal, "ANN MSE (Calculated)": round(mse_calc, 2)})
    print(f"Galaxy {gal:8s} | Calculated ANN MSE: {mse_calc:.2f}")

summary_df = pd.DataFrame(results_list)
print("
Summary Table:")
print(summary_df.to_string(index=False))
